# Notebook 03 — NMF Recovery

**Residue Manifold Learning**

This notebook tests whether nonnegative matrix factorization (NMF) can recover mod30 residue-lane structure from sampled modular data.

Notebook 01 established that prime residues occupy 8 valid lanes modulo 30.  
Notebook 02 showed that constrained sampling improves access to those lanes.  
Notebook 03 asks whether the structure can be learned automatically from residue-count matrices.

**Core claim:**

> NMF recovers constrained residue-lane structure as sparse nonnegative components over mod30 residue classes.

## 1. Setup

Figures are saved as SVG only. Do not save PNG duplicates.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error

mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["figure.dpi"] = 120

MOD = 30
VALID_LANES_MOD30 = [1, 7, 11, 13, 17, 19, 23, 29]
SEED = 9423

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

print("Valid mod30 lanes:", VALID_LANES_MOD30)

## 2. Helper functions

The batch matrix has shape:

```text
n_batches × 30 residue classes
```

Each row is a normalized residue-count vector from one sampled batch.

In [ ]:
def save_svg(fig, name):
    """Save a Matplotlib figure as canonical SVG output."""
    path = f"figures/{name}.svg"
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")


def make_batch_matrix(
    n_batches=400,
    batch_size=250,
    constrained=True,
    seed=SEED,
    n_max=100_000,
):
    """Create a residue-count matrix from sampled integers.

    If constrained=True, samples only from valid mod30 lanes.
    If constrained=False, samples uniformly from all integers in [2, n_max).
    """
    rng = np.random.default_rng(seed)
    all_numbers = np.arange(2, n_max)
    valid_numbers = all_numbers[np.isin(all_numbers % MOD, VALID_LANES_MOD30)]

    rows = []
    for _ in range(n_batches):
        if constrained:
            sample = rng.choice(valid_numbers, size=batch_size, replace=True)
        else:
            sample = rng.choice(all_numbers, size=batch_size, replace=True)
        counts = np.bincount(sample % MOD, minlength=MOD)
        rows.append(counts)

    return np.asarray(rows)


def row_normalize(X):
    """Normalize rows to sum to one."""
    denom = X.sum(axis=1, keepdims=True)
    denom[denom == 0] = 1
    return X / denom


def lane_mass_ratio(component, valid_lanes=VALID_LANES_MOD30, mod=MOD):
    """Fraction of component mass that falls on valid mod30 lanes."""
    total = component.sum()
    if total <= 0:
        return 0.0
    mask = np.zeros(mod, dtype=bool)
    mask[valid_lanes] = True
    return float(component[mask].sum() / total)


def component_peak_residue(component):
    """Residue class with maximum component mass."""
    return int(np.argmax(component))

## 3. Generate input matrices

We generate two residue matrices:

- `Xc`: constrained samples from valid mod30 lanes
- `Xu`: uniform samples from all residue classes

The NMF recovery experiment focuses on `Xc` first, because Notebook 03 is the positive recovery baseline.

In [ ]:
X_constrained_counts = make_batch_matrix(constrained=True, seed=SEED)
X_uniform_counts = make_batch_matrix(constrained=False, seed=SEED + 1)

Xc = row_normalize(X_constrained_counts)
Xu = row_normalize(X_uniform_counts)

print("Xc shape:", Xc.shape)
print("Xu shape:", Xu.shape)
print("Constrained row sum check:", Xc[0].sum())
print("Uniform row sum check:", Xu[0].sum())

## 4. Figure — NMF input matrix

The constrained input matrix should show only 8 active vertical lanes.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(Xc[:120], aspect="auto", interpolation="nearest")
ax.set_title("NMF Input Matrix: Constrained mod30 Residue Counts")
ax.set_xlabel("Residue class mod 30")
ax.set_ylabel("Sample batch")
ax.set_xticks(range(MOD))
ax.set_xticklabels(range(MOD), fontsize=7)
fig.colorbar(im, ax=ax, label="Normalized count")
fig.tight_layout()
save_svg(fig, "nmf_input_matrix")
plt.show()

## 5. Run NMF with 8 components

Because the mod30 prime-support structure has 8 valid lanes, the baseline factorization uses `n_components = 8`.

In [ ]:
n_components = 8

model = NMF(
    n_components=n_components,
    init="nndsvda",
    random_state=SEED,
    max_iter=2000,
)

W = model.fit_transform(Xc)
H = model.components_
X_hat = W @ H

reconstruction_mse = mean_squared_error(Xc, X_hat)
print("Reconstruction MSE:", reconstruction_mse)
print("W shape:", W.shape)
print("H shape:", H.shape)

## 6. Figure — Learned NMF components

Rows are learned components. Columns are residue classes mod30.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(H, aspect="auto", interpolation="nearest")
ax.set_title("NMF Components over mod30 Residue Classes")
ax.set_xlabel("Residue class mod 30")
ax.set_ylabel("Component")
ax.set_xticks(range(MOD))
ax.set_xticklabels(range(MOD), fontsize=7)
ax.set_yticks(range(n_components))
ax.set_yticklabels([f"C{i}" for i in range(n_components)])
fig.colorbar(im, ax=ax, label="Component mass")
fig.tight_layout()
save_svg(fig, "nmf_components")
plt.show()

## 7. Component alignment with valid lanes

A learned component is aligned with the residue manifold when most of its mass falls on the 8 valid lanes.

In [ ]:
component_records = []
for i, h in enumerate(H):
    component_records.append({
        "component": i,
        "peak_residue": component_peak_residue(h),
        "lane_mass_ratio": lane_mass_ratio(h),
        "component_total_mass": float(h.sum()),
    })

df_component_summary = pd.DataFrame(component_records)
df_component_summary

## 8. Figure — Lane alignment score

The lane-mass ratio measures how much of each learned component lies on the true mod30 residue manifold.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_component_summary["component"], df_component_summary["lane_mass_ratio"])
ax.axhline(1.0, linestyle="--", linewidth=1)
ax.set_ylim(0, 1.05)
ax.set_title("NMF Component Alignment with Valid mod30 Lanes")
ax.set_xlabel("Component")
ax.set_ylabel("Lane mass ratio")
ax.set_xticks(df_component_summary["component"])
fig.tight_layout()
save_svg(fig, "nmf_lane_alignment")
plt.show()

## 9. Reconstruction sweep

We sweep component count `k = 1..12` to see how reconstruction error changes as model capacity increases.

This is a precursor to later capacity/coverage comparisons.

In [ ]:
records = []

for k in range(1, 13):
    model_k = NMF(
        n_components=k,
        init="nndsvda",
        random_state=SEED,
        max_iter=2000,
    )
    Wk = model_k.fit_transform(Xc)
    Hk = model_k.components_
    Xhk = Wk @ Hk

    mse = mean_squared_error(Xc, Xhk)
    ratios = [lane_mass_ratio(h) for h in Hk]
    peak_residues = [component_peak_residue(h) for h in Hk]
    unique_valid_peaks = len(set(peak_residues).intersection(VALID_LANES_MOD30))

    records.append({
        "k": k,
        "reconstruction_mse": mse,
        "mean_lane_mass_ratio": float(np.mean(ratios)),
        "min_lane_mass_ratio": float(np.min(ratios)),
        "unique_valid_peak_residues": unique_valid_peaks,
    })

df_summary = pd.DataFrame(records)
df_summary

## 10. Figure — Reconstruction error

Error should decrease as component capacity increases. The mod30 lane count provides a natural reference point at `k = 8`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(df_summary["k"], df_summary["reconstruction_mse"], marker="o")
ax.axvline(8, linestyle="--", linewidth=1, label="8 valid lanes")
ax.set_title("NMF Reconstruction Error vs Component Count")
ax.set_xlabel("Number of NMF components (k)")
ax.set_ylabel("Reconstruction MSE")
ax.set_xticks(df_summary["k"])
ax.legend()
fig.tight_layout()
save_svg(fig, "nmf_reconstruction_error")
plt.show()

## 11. Save outputs

Save summary tables and learned components for later notebooks and paper figures.

In [ ]:
df_summary.to_csv("data/nmf_recovery_summary.csv", index=False)
df_component_summary.to_csv("data/nmf_component_summary.csv", index=False)

df_components = pd.DataFrame(H, columns=[f"residue_{r}" for r in range(MOD)])
df_components.insert(0, "component", np.arange(H.shape[0]))
df_components["peak_residue"] = [component_peak_residue(h) for h in H]
df_components["lane_mass_ratio"] = [lane_mass_ratio(h) for h in H]
df_components.to_csv("data/nmf_components.csv", index=False)

print("Saved:")
print("- data/nmf_recovery_summary.csv")
print("- data/nmf_component_summary.csv")
print("- data/nmf_components.csv")

## 12. Notebook 03 claim

> NMF recovers the constrained residue manifold as sparse nonnegative components over mod30 lanes, showing that modular structure can be learned from samples rather than manually specified.

Notebook 04 should introduce the contrasting case: sparse autoencoder dilution and fragmentation.

## 13. Optional download bundle

Uncomment the final two lines to trigger a browser download in Colab.

In [ ]:
# --- Optional: Download outputs (uncomment last line to trigger) ---

import os
import zipfile

zip_name = "03_nmf_recovery_outputs.zip"
folders_to_zip = ["data", "figures"]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders_to_zip:
        if os.path.exists(folder):
            for root, _, filenames in os.walk(folder):
                for filename in filenames:
                    path = os.path.join(root, filename)
                    z.write(path, arcname=path)

print(f"Prepared: {zip_name}")

# from google.colab import files
# files.download(zip_name)